In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

In [2]:
train = pd.read_csv("data/dataset/train.csv")
test = pd.read_csv("data/dataset/test.csv")
sample = pd.read_csv("data/dataset/sample_submission.csv")

print(train.shape)
print(test.shape)
print(sample.shape)

(77299, 11)
(41778, 10)
(5, 2)


In [3]:
print(train.columns)
print(test.columns)
print(sample.columns)

sample.head()

Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
       'Weather'],
      dtype='object')
Index(['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'],
      dtype='object')
Index(['Index', 'demand'], dtype='object')


,Index,demand
0,0,0.090768
1,1,0.089885
2,2,0.007037
3,3,0.079087
4,4,0.054636


In [4]:
X = train.drop("demand", axis=1)
y = train["demand"]

In [5]:
for df in [X, test]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%H:%M")

    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute

    df.drop("timestamp", axis=1, inplace=True)

In [6]:
combined = pd.concat([X, test], axis=0)

for col in combined.columns:
    if combined[col].dtype == "object":
        le = LabelEncoder()
        combined[col] = le.fit_transform(
            combined[col].astype(str)
        )

X = combined.iloc[:len(train)]
test_processed = combined.iloc[len(train):]

In [7]:
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
preds = model.predict(test_processed)

print(preds[:10])

[0.08847895 0.03368999 0.03777279 0.03172237 0.03859108 0.05482842
 0.02407392 0.06201015 0.0230911  0.07116249]


In [9]:
sample.head()

,Index,demand
0,0,0.090768
1,1,0.089885
2,2,0.007037
3,3,0.079087
4,4,0.054636


In [10]:
submission = sample.copy()

submission["demand"] = preds

submission.to_csv(
    "submission_v1.csv",
    index=False
)

submission.head()

ValueError: Length of values (41778) does not match length of index (5)

In [11]:
print(sample.shape)
print(sample.columns)
print(sample.head())

(5, 2)
Index(['Index', 'demand'], dtype='object')
   Index    demand
0      0  0.090768
1      1  0.089885
2      2  0.007037
3      3  0.079087
4      4  0.054636


In [12]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": preds
})

print(submission.shape)
print(submission.head())

submission.to_csv("submission_v1.csv", index=False)

(41778, 2)
   Index    demand
0      0  0.088479
1      1  0.033690
2      2  0.037773
3      3  0.031722
4      4  0.038591


In [13]:
check = pd.read_csv("submission_v1.csv")

print(check.shape)
print(check.columns)
print(check.head())

(41778, 2)
Index(['Index', 'demand'], dtype='object')
   Index    demand
0      0  0.088479
1      1  0.033690
2      2  0.037773
3      3  0.031722
4      4  0.038591


In [14]:
assert submission.shape == (41778, 2)
assert list(submission.columns) == ["Index", "demand"]

print("Submission format valid")

Submission format valid
